In [ ]:
import pandas as pd
import numpy as np
import time


In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [16]:
# 自动识别ROCm GPU / CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前使用计算设备：{device}")
# ROCm环境下，会输出：当前使用计算设备：cuda
# 本地无GPU电脑，输出：当前使用计算设备：cpu

当前使用计算设备：cuda


In [17]:
# 加载数据集
df = pd.read_csv("./mp_20k.csv")  
df = df.dropna(subset=["e_form","bandgap"])
print("有效样本数量：", len(df))

有效样本数量： 20000


In [18]:
# 简易特征构造：统计元素数量
def count_elements(formula):
    import re
    els = re.findall(r'[A-Z][a-z]*', formula)
    return len(set(els))

df["n_ele"] = df["formula"].apply(count_elements)

In [19]:
# 构造输入特征与标签
X = df[["n_ele"]].values.astype(np.float32)
Y = df[["e_form","bandgap"]].values.astype(np.float32)

In [20]:
# 划分训练集、测试集
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

In [21]:
# 转为tensor
X_train_tensor = torch.from_numpy(X_train)
Y_train_tensor = torch.from_numpy(Y_train)
X_test_tensor = torch.from_numpy(X_test)
Y_test_tensor = torch.from_numpy(Y_test)

In [24]:
#构建数据集类与 MLP 模型
class MaterialDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __len__(self):
        return len(self.x)
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

train_ds = MaterialDataset(X_train_tensor, Y_train_tensor)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)

# MLP模型，输入特征→预测形成能、带隙2个物性
class MLP(nn.Module):
    def __init__(self, in_dim=1, hidden_dim=64, out_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim)
        )
    def forward(self, x):
        return self.net(x)

model = MLP().to(device)
loss_fn = nn.MSELoss()
opt = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
#模型训练，计时
model.train()
epoch_num = 50
start_time = time.time()

for epoch in range(epoch_num):
    total_loss = 0.0
    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        pred = model(x_batch)
        loss = loss_fn(pred, y_batch)

        opt.zero_grad()
        loss.backward()
        opt.step()
        total_loss += loss.item()
    if (epoch+1)%10 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {total_loss:.4f}")

end_time = time.time()
train_time = end_time - start_time
print(f"\n训练总耗时：{train_time:.2f} 秒")

Epoch  10 | Loss: 71.0234
Epoch  20 | Loss: 71.1491
Epoch  30 | Loss: 70.8767
Epoch  40 | Loss: 71.0593
Epoch  50 | Loss: 71.0539

训练总耗时：3.65 秒


In [ ]:
#模型评估与结果可视化
model.eval()
with torch.no_grad():
    X_test_dev = X_test_tensor.to(device)
    Y_pred = model(X_test_dev).cpu().numpy()

# 计算MAE，评估预测精度
from sklearn.metrics import mean_absolute_error
mae_eform = mean_absolute_error(Y_test[:,0], Y_pred[:,0])
mae_bandgap = mean_absolute_error(Y_test[:,1], Y_pred[:,1])
print(f"形成能预测MAE: {mae_eform:.4f}")
print(f"带隙预测MAE: {mae_bandgap:.4f}")

形成能预测MAE: 0.7897
带隙预测MAE: 0.8846
